# Analyse the Full Workflow Executions - Highlight Key Data

In [11]:
from tabulate import tabulate
import pandas as pd

In [12]:
def joules_to_kwh(joules):
    return float(joules) / 3600000

def get_error(experimental, actual):
    return round((abs(experimental - actual) / actual) * 100, 2)

In [13]:
# Functions to parse rapl data
def load_workflow_rapl_readings(cluster, workflow):
    with open(f'{cluster}-cluster/{workflow}-runs.csv', 'r') as file:
        data = [line.strip().split(',') for line in file.readlines()]
    
    return data[1:]

def process_rapl_data(data):
    processed = {}
    run = 1

    for row in data:
        processed[run] = {}
        processed[run]['pkg'] = float(row[-3])
        processed[run]['dram'] = float(row[-2])
        processed[run]['total'] = float(row[-1])
        run += 1

    return processed

In [14]:
# Functions to parse ichnos data
def load_ichnos_summary_file(summary_file):
    with open(summary_file, 'r') as file:
        raw = [line.strip() for line in file.readlines()]

    cpu = float(raw[7].split(':')[1].strip()[:-3])
    task_mem = float(raw[9].split(':')[1].strip()[:-3])
    node_mem = float(raw[15].split(':')[1].strip()[:-3])

    return (cpu, task_mem, node_mem)

def load_ichnos_data(cluster, workflow, runs, strategy):
    workflow_path = f'{cluster}-cluster/{workflow}'
    data = {}

    for run in range(1, runs + 1):
        data[run] = {}
        (cpu, task_mem, node_mem) = load_ichnos_summary_file(f'{workflow_path}/{run}/{cluster}-{workflow}-{run}-1-{strategy}-summary.txt')
        data[run]['pkg'] = cpu
        data[run]['dram'] = task_mem + node_mem 
        data[run]['total'] = cpu + task_mem + node_mem

    return data

In [15]:
print('Cluster Workflow Executions')

rapl_data = {'hu': {}, 'gu': {}}
rapl_data['hu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'rnaseq'))
rapl_data['hu']['chipseq'] = process_rapl_data(load_workflow_rapl_readings('hu', 'chipseq'))
rapl_data['gu']['rnaseq'] = process_rapl_data(load_workflow_rapl_readings('gu', 'rnaseq'))
rapl_data['hu']['rangeland'] = process_rapl_data(load_workflow_rapl_readings('hu', 'rangeland'))

table_data = []
table_head = ['cluster', 'workflow', 'run', 'pkg (kWh)', 'dram (kWh)', 'total (kWh)']

for cluster in ['hu', 'gu']:
    for workflow in rapl_data[cluster].keys():
        for run in range(1, len(rapl_data[cluster][workflow].keys()) + 1):
            rapl_entry = rapl_data[cluster][workflow][run]
            table_data.append([cluster, workflow, run, round(rapl_entry['pkg'], 2), round(rapl_entry['dram'], 2), round(rapl_entry['total'], 2)])

print(tabulate(table_data, table_head))

Cluster Workflow Executions
cluster    workflow      run    pkg (kWh)    dram (kWh)    total (kWh)
---------  ----------  -----  -----------  ------------  -------------
hu         rnaseq          1         1.82          0.45           2.27
hu         rnaseq          2         1.82          0.51           2.32
hu         rnaseq          3         1.25          0.27           1.51
hu         chipseq         1         3.13          0.73           3.85
hu         chipseq         2         3.13          0.73           3.86
hu         chipseq         3         2.42          0.67           3.09
hu         rangeland       1         0.3           0.98           1.28
hu         rangeland       2         0.37          0.12           0.49
hu         rangeland       3         0.3           0.25           0.55
gu         rnaseq          1         1.15          0.24           1.39
